<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Find the unknown unknowns in your OpenRouter traffic with Langfuse Topics" sidebarTitle: "Topics with OpenRouter" description: "Call OpenRouter models through the Langfuse AI Gateway, group the traces into named topics with Langfuse Topics, build a dataset from one topic, and compare OpenRouter models on it." category: "Examples" -->

# Find the unknown unknowns in your OpenRouter traffic with Langfuse Topics

[OpenRouter](https://openrouter.ai/) tells you which model handled each request. This cookbook shows how to find out what those requests are about, where they go wrong, and which model you should use for them with [Langfuse](https://langfuse.com).

You will:

1. Run a customer support agent that calls OpenRouter models through the [Langfuse AI Gateway](https://langfuse.com/docs/gateway/overview). The gateway records every model call in Langfuse as traces.
2. Let [Langfuse Topics](https://langfuse.com/docs/observability/features/topics) cluster your traces into named topics, so you can see how your agent is used and where it breaks.
3. Find two unknown unknowns: a request that the assistant could handle itself with one more tool, and a failure that is spread over many kinds of requests.
4. Collect the traces of the failure into a [dataset](https://langfuse.com/docs/evaluation/experiments/datasets), run it against several OpenRouter models as [experiments](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk), and compare quality, cost and latency.

You can also use [OpenRouter](https://langfuse.com/integrations/gateways/openrouter) via Broadcast or SDK.

## 1. Setup

Install the dependencies, then set your keys in the next cell.

You need three sets of keys:

- The **Langfuse API keys** of the project that the gateway records into. The notebook uses them to read the traces, and for the dataset and the experiments.
- `LANGFUSE_GATEWAY_KEY` is the key the agent calls the gateway with. You create it in section 3.
- `OPENROUTER_API_KEY` is used by the simulated customers. Their calls go to OpenRouter directly, so they are not recorded in Langfuse.

In [ ]:
%pip install -q "langfuse>=5" "openai>=3" pandas requests

In [ ]:
import os

# Langfuse: Settings → API Keys, in the project that the AI Gateway records into
os.environ.setdefault("LANGFUSE_PUBLIC_KEY", "pk-lf-...")
os.environ.setdefault("LANGFUSE_SECRET_KEY", "sk-lf-...")
os.environ.setdefault("LANGFUSE_BASE_URL", "https://cloud.langfuse.com")  # 🇪🇺 EU region
# Other Langfuse data regions include 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com and ⚕️ HIPAA: https://hipaa.cloud.langfuse.com

# Langfuse AI Gateway: AI Gateway → Gateway keys, see section 3
os.environ.setdefault("LANGFUSE_GATEWAY_KEY", "...")
os.environ.setdefault("LANGFUSE_GATEWAY_HOST", "https://gateway.cloud.langfuse.com")  # 🇪🇺 EU region
# Other regions: 🇺🇸 US: https://gateway.us.cloud.langfuse.com, 🇯🇵 Japan: https://gateway.jp.cloud.langfuse.com and ⚕️ HIPAA: https://gateway.hipaa.cloud.langfuse.com

# OpenRouter: https://openrouter.ai/settings/keys
os.environ.setdefault("OPENROUTER_API_KEY", "sk-or-...")  # simulated customers
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [ ]:
from langfuse import get_client
from openai import OpenAI

langfuse = get_client()
assert langfuse.auth_check(), "Langfuse auth failed. Check your keys and base URL."

# The gateway speaks the OpenAI API, so the agent uses the OpenAI SDK with the gateway as its base URL.
app_client = OpenAI(base_url=os.environ["LANGFUSE_GATEWAY_HOST"] + "/openai/v1", api_key=os.environ["LANGFUSE_GATEWAY_KEY"])

## 2. The sample agent

The agent is a support assistant of a fictional company that sells card readers and a point-of-sale app to small shops.

It has five tools: `search_help_center`, `get_customer`, `check_integration`, `issue_refund` and `handoff_to_human`.

The tools run locally on synthetic data. The setup has a few flaws built in, of the kind that real systems collect over time: the help center still holds an outdated pricing article, the refund tool cannot refund invoices that were paid by bank transfer, and one rule in the system prompt asks for the invoice number even when the customer already gave it. Nothing in the traces labels these flaws.

<details>
<summary>Demo data: customers, invoices, orders, integrations and the help center</summary>

In [ ]:
import itertools
import json
import random
import re
from datetime import date, timedelta

TODAY = date.today()
data_rng = random.Random(7)

PLAN_PRICES = {"Starter": 29, "Plus": 79, "Pro": 149}  # USD per month
OLD_PLAN_PRICES = {"Starter": 19, "Plus": 59, "Pro": 119}  # until the price change last month
INTEGRATION_ERRORS = {
    "ledgerly": ["LEDGERLY_TOKEN_EXPIRED", "LEDGERLY_TAX_MAPPING_MISSING"],  # accounting
    "shopfront": ["SHOPFRONT_PLUGIN_OUTDATED", "SHOPFRONT_SKU_CONFLICT"],  # online store
    "webhooks": ["WEBHOOK_SIGNATURE_MISMATCH"],
}
OWNERS = ["Maya Lindqvist", "Jonas Weber", "Priya Raman", "Tomas Keller", "Lucia Ferrand", "Daniel Osei",
          "Akira Sato", "Elena Petrova", "Marcus Hale", "Sofia Marin", "Noah Brandt", "Ines Duarte"]
SHOPS = ["Blue Heron", "Alder and Oak", "Third Rail", "Marigold", "Copper Kettle", "Saltbox", "Lantern", "Field Day",
         "Paper Crane", "Driftwood", "Kiln", "Morning Tide", "Juno Street", "Ember and Ash", "Quiet Corner", "Fern"]
SHOP_TYPES = ["Coffee", "Bakery", "Books"]
ITEMS = ["Tallybird Reader 2", "Tallybird Reader 2 with counter stand", "Receipt printer P1"]


def month_start(months_back: int) -> date:
    year, month = TODAY.year, TODAY.month - months_back
    while month < 1:
        year, month = year - 1, month + 12
    return date(year, month, 1)


def integration_state(integration: str, plan: str) -> dict:
    if integration == "webhooks" and plan == "Starter":
        return {"status": "unavailable", "error": "WEBHOOK_PLAN_REQUIRED"}
    status = data_rng.choices(["not_connected", "connected", "error"], [4, 2, 4])[0]
    if status != "error":
        return {"status": status}
    return {"status": "error", "error": data_rng.choice(INTEGRATION_ERRORS[integration]),
            "last_successful_sync": (TODAY - timedelta(days=data_rng.randint(2, 12))).isoformat()}


CUSTOMERS = {}
for number in range(48):
    customer_id = f"cus_{1001 + number}"
    plan = data_rng.choices(list(PLAN_PRICES), [5, 4, 2])[0]
    invoices = [
        {"invoice_id": f"INV-{48200 + number * 4 + position}", "date": month_start(months_back).isoformat(),
         "amount_usd": (OLD_PLAN_PRICES if months_back == 2 else PLAN_PRICES)[plan], "status": "paid"}
        for position, months_back in enumerate((2, 1, 0))
    ]
    if number % 3 == 0:  # charged twice this month
        invoices.append({**invoices[-1], "invoice_id": f"INV-{48200 + number * 4 + 3}"})
    status = data_rng.choices(["processing", "shipped", "delayed", "delivered"], [2, 4, 3, 3])[0]
    order = {"order_id": f"ORD-{70310 + number}", "item": data_rng.choice(ITEMS), "status": status,
             "ordered": (TODAY - timedelta(days=data_rng.randint(6, 12))).isoformat()}
    if status != "processing":
        order["carrier"], order["tracking_number"] = "Northpost", f"NP{data_rng.randint(10**8, 10**9 - 1)}"
        delivery_offset = {"shipped": data_rng.randint(1, 4), "delayed": -data_rng.randint(2, 5), "delivered": -data_rng.randint(1, 3)}[status]
        order["delivered" if status == "delivered" else "expected_delivery"] = (TODAY + timedelta(days=delivery_offset)).isoformat()
    CUSTOMERS[customer_id] = {
        "customer_id": customer_id,
        "name": OWNERS[number % len(OWNERS)],
        "business": f"{SHOPS[number % len(SHOPS)]} {SHOP_TYPES[number // len(SHOPS)]}",
        "plan": plan,
        "payment_method": data_rng.choices(["card", "bank transfer"], [55, 45])[0],
        "invoices": invoices,
        "order": order,
        "integrations": {name: integration_state(name, plan) for name in INTEGRATION_ERRORS},
    }

HELP_CENTER = {
    "Find and download invoices": "Open Settings > Billing > Invoices. Every invoice has a Download PDF button. Invoices are also emailed to the billing contact on the first day of each month.",
    "Add a VAT ID to your invoices": "Enter your VAT ID under Settings > Billing > Company details. It appears on all future invoices. To get a past invoice reissued with the VAT ID, contact support.",
    "Change your payment method": "Open Settings > Billing > Payment method. You can pay by card or by bank transfer. The new method is used from the next invoice.",
    "Plans and pricing": "Starter costs 19 USD per month, Plus 59 USD per month and Pro 119 USD per month. Starter includes one register. Plus adds webhooks, the API and three registers. Pro adds unlimited registers and priority support.",
    "Price change for all plans": f"Since {month_start(1):%B %Y}, the prices are 29 USD per month for Starter, 79 USD for Plus and 149 USD for Pro. Paying yearly saves 15 percent. The plans include the same features as before.",
    "Refunds": "A duplicate charge is refunded in full. Other charges can be refunded within 14 days if the plan was not used in that period. Refunds go back to the original payment method within 5 to 10 business days.",
    "Reset your password": "Select Forgot password on the sign-in page. The reset link is sent to the email address of the account and is valid for 30 minutes. If no email arrives within 10 minutes, check the spam folder and that you entered the address of the account owner. Accounts that sign in with single sign-on reset the password at their identity provider.",
    "Locked out after failed sign-ins": "After five failed sign-in attempts the account is locked for 15 minutes. Wait, then sign in again or reset your password. Support cannot lift the lock earlier.",
    "Create and rotate API keys": "API keys are managed under Settings > Developers > API keys, on the Plus and Pro plans. When you rotate a key, the old key keeps working for 24 hours. If a key was exposed, select Revoke now instead: the key stops working at once.",
    "Available integrations": "Tallybird connects to Ledgerly for accounting and to Shopfront for online stores. Other systems can be connected with webhooks and the API on the Plus and Pro plans. There are no other built-in integrations.",
    "Connect Ledgerly": "Open Settings > Integrations > Ledgerly, select Connect and sign in to Ledgerly. Daily sales totals are synced every night.",
    "Connect Shopfront": "Install the Tallybird plugin in your Shopfront store, copy the store key from the plugin and paste it under Settings > Integrations > Shopfront. Products and stock levels sync both ways.",
    "Set up webhooks": "Open Settings > Developers > Webhooks, add your endpoint URL and choose the events. Each request is signed with the signing secret shown on that page.",
    "Error LEDGERLY_TOKEN_EXPIRED": "The connection to Ledgerly expires after 90 days. Open Settings > Integrations > Ledgerly and select Reconnect. Sales from the days that were missed are synced automatically, up to 30 days back.",
    "Error LEDGERLY_TAX_MAPPING_MISSING": "The first sync cannot start before each Tallybird tax rate is mapped to a Ledgerly tax code. Open Settings > Integrations > Ledgerly > Tax mapping, map every rate and select Sync now.",
    "Error SHOPFRONT_PLUGIN_OUTDATED": "Stock sync needs version 3.2 or later of the Tallybird plugin. Update the plugin in your Shopfront admin, then enter the store key again under Settings > Integrations > Shopfront.",
    "Error SHOPFRONT_SKU_CONFLICT": "The product import stops when two products share a SKU. Download the conflict list under Settings > Integrations > Shopfront > Import log, make the SKUs unique in Shopfront and run the import again.",
    "Error WEBHOOK_SIGNATURE_MISMATCH": "Compute the signature over the raw request body, before any JSON parsing, with the signing secret from Settings > Developers > Webhooks. The signing secret is not the API key, and rotating the API key does not change it.",
    "Error WEBHOOK_PLAN_REQUIRED": "Webhooks are part of the Plus and Pro plans. On Starter, the Webhooks page is hidden. Upgrade under Settings > Billing > Plan to use them.",
    "Cancel your subscription": "Open Settings > Billing > Plan and select Cancel plan. The plan stays active until the end of the paid month. Your sales data is kept for 90 days after that and can be exported under Settings > Data export.",
    "Pause your subscription": "Instead of cancelling, you can pause for up to three months under Settings > Billing > Plan. A paused plan costs nothing and keeps all data.",
    "Shipping times": "Hardware ships within 2 business days and arrives 3 to 5 business days later. You get a tracking link from Northpost by email when the order ships.",
    "Late or missing deliveries": "If an order is more than 3 business days past its expected delivery, or is marked as delivered but did not arrive, the support team sends a replacement. The assistant cannot do this itself.",
    "Change the delivery address": "The address can be changed under Orders while the order is still in processing. Once it has shipped, contact Northpost with the tracking number.",
    "Return hardware": "Hardware can be returned within 30 days of delivery. Start the return under Orders to get a prepaid label.",
}
print(len(CUSTOMERS), "customers,", len(HELP_CENTER), "help center articles")

</details>

The tools are plain Python functions. Each returns a string, including when it fails, so the model sees the error and can react to it.

<details>
<summary>Tools: the five functions and their schemas</summary>

In [ ]:
TICKET_NUMBERS = itertools.count(20481)
STOPWORDS = {"the", "and", "for", "our", "what", "how", "can", "are", "you", "with", "from", "does", "your",
             "about", "get", "need", "should", "customer", "their", "they", "this", "that", "have", "when", "tallybird"}


def search_help_center(query: str) -> str:
    words = {word for word in re.findall(r"[a-z0-9]+", query.lower()) if len(word) > 2} - STOPWORDS
    scored = []
    for title, text in HELP_CENTER.items():
        article_words = set(re.findall(r"[a-z0-9]+", f"{title} {text}".lower()))
        scored.append((len(words & article_words), f"{title}: {text}"))
    hits = [article for score, article in sorted(scored, reverse=True)[:2] if score > 0]
    return "\n\n".join(hits) or "No matching article found."


def get_customer(customer_id: str) -> str:
    customer = CUSTOMERS.get(customer_id)
    if customer is None:
        return f"Error: no customer with ID '{customer_id}'."
    return json.dumps({key: value for key, value in customer.items() if key != "integrations"})


def check_integration(customer_id: str, integration: str) -> str:
    customer = CUSTOMERS.get(customer_id)
    if customer is None:
        return f"Error: no customer with ID '{customer_id}'."
    state = customer["integrations"].get(integration.strip().lower())
    if state is None:
        return f"Error: unknown integration '{integration}'. Available: {', '.join(customer['integrations'])}."
    return json.dumps({"integration": integration.strip().lower(), **state})


def issue_refund(customer_id: str, invoice_id: str) -> str:
    customer = CUSTOMERS.get(customer_id)
    if customer is None:
        return f"Error: no customer with ID '{customer_id}'."
    invoice = next((invoice for invoice in customer["invoices"] if invoice["invoice_id"] == invoice_id.strip()), None)
    if invoice is None:
        return f"Error: customer {customer_id} has no invoice '{invoice_id}'."
    if customer["payment_method"] != "card":
        return "Error: refund_failed. The payment provider rejected the request: payment method not supported."
    return f"Refunded {invoice['amount_usd']} USD for {invoice['invoice_id']} to the customer's card."


def handoff_to_human(customer_id: str, summary: str) -> str:
    return f"Ticket TCK-{next(TICKET_NUMBERS)} opened. The support team replies by email within one business day."


def tool_schema(function, description, **parameters):
    return {"type": "function", "function": {
        "name": function.__name__, "description": description,
        "parameters": {"type": "object", "required": list(parameters),
                       "properties": {name: {"type": "string", "description": text} for name, text in parameters.items()}},
    }}


CUSTOMER_ID = "ID of the signed-in customer, for example cus_1001"
TOOL_FUNCTIONS = {f.__name__: f for f in (search_help_center, get_customer, check_integration, issue_refund, handoff_to_human)}
TOOLS = [
    tool_schema(search_help_center, "Search the help center. Returns the two best matching articles.", query="Keywords or an error code"),
    tool_schema(get_customer, "Get the customer's plan, payment method, invoices and hardware order.", customer_id=CUSTOMER_ID),
    tool_schema(check_integration, "Get the status and the last error of one of the customer's integrations.",
                customer_id=CUSTOMER_ID, integration="One of: ledgerly, shopfront, webhooks"),
    tool_schema(issue_refund, "Refund one invoice to the customer's payment method.", customer_id=CUSTOMER_ID, invoice_id="Invoice ID, for example INV-48200"),
    tool_schema(handoff_to_human, "Open a ticket for the human support team.", customer_id=CUSTOMER_ID, summary="What the customer needs and what was tried"),
]


def call_tool(name: str, arguments: str) -> str:
    try:
        return TOOL_FUNCTIONS[name](**json.loads(arguments or "{}"))
    except Exception as error:  # unknown tool, malformed or missing arguments
        return f"Tool error: {error!r}"

</details>

The agent is a plain tool-calling loop on the OpenAI SDK. `agent_reply` answers the latest customer message: it lets the model call tools for up to four rounds, then the model has to reply with what it has. The whole conversation lives in `messages`, so the next customer message continues where the last reply ended.

`agent_reply` contains no tracing code, because the gateway records the model calls. It calls the model with `client.chat.completions.create`, and the default model is `openrouter:liquid/lfm-3-1.7b`. Request options, such as the headers that the gateway reads, are passed through `request_options`.

<details>
<summary>Agent: the system prompt and the tool-calling loop</summary>

In [ ]:
DEFAULT_MODEL = "openrouter:liquid/lfm-3-1.7b"  # <name of the gateway connection>:<OpenRouter model ID>, see section 3
MAX_STEPS = 5

SYSTEM_PROMPT = """You are the support assistant of Tallybird, which sells card readers and a point-of-sale app to small shops. Today is {today}.
The customer is signed in as {name} of {business}, customer ID {customer_id}.

- Take product facts, steps and prices from the help center only: use search_help_center. If the help center does not cover something, say so.
- For questions about the customer's plan, invoices or hardware order, use get_customer.
- For a problem with an integration, call check_integration before you answer.
- Before you issue a refund, ask the customer to confirm the invoice number.
- Hand off to a human when the customer asks for one, or when you cannot solve the request yourself.
- You only help with Tallybird. Politely decline everything else.
- Keep replies short and friendly."""


def new_conversation(customer_id: str) -> list[dict]:
    customer = CUSTOMERS[customer_id]
    prompt = SYSTEM_PROMPT.format(today=TODAY.isoformat(), **{key: customer[key] for key in ("name", "business", "customer_id")})
    return [{"role": "system", "content": prompt}]


def agent_reply(messages: list[dict], *, client: OpenAI, model: str = DEFAULT_MODEL, **request_options) -> float:
    """Answer the latest customer message. Appends tool calls, tool results and the reply to `messages`, returns the cost in USD."""
    cost = 0.0
    for step in range(MAX_STEPS):
        last_step = step == MAX_STEPS - 1
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=TOOLS,
            tool_choice="none" if last_step else "auto",
            **request_options,
        )
        cost += getattr(response.usage, "cost", None) or 0.0  # OpenRouter returns the cost of each call
        message = response.choices[0].message
        if not message.tool_calls or last_step:
            messages.append({"role": "assistant", "content": message.content or ""})
            break
        messages.append({"role": "assistant", "content": message.content or "",
                         "tool_calls": [call.model_dump(include={"id", "type", "function"}) for call in message.tool_calls]})
        for call in message.tool_calls:
            messages.append({"role": "tool", "tool_call_id": call.id,
                             "content": call_tool(call.function.name, call.function.arguments)})
    return cost


def tool_log(messages: list[dict]) -> list[dict]:
    """The tool calls of a conversation with their results."""
    results = {message["tool_call_id"]: message["content"] for message in messages if message["role"] == "tool"}
    return [{"tool": call["function"]["name"], "arguments": call["function"]["arguments"], "result": results.get(call["id"], "")}
            for message in messages for call in message.get("tool_calls", [])]

</details>

## 3. Call OpenRouter through the Langfuse AI Gateway

<!-- CALLOUT_START type: "info" emoji: "ℹ️" -->
The Langfuse AI Gateway is in beta.
<!-- CALLOUT_END -->

The [Langfuse AI Gateway](https://langfuse.com/docs/gateway/overview) sits between your application and the model provider. Your application sends its requests to the gateway with a gateway key. The gateway forwards them to OpenRouter with the OpenRouter key it holds, and records every call as a generation in Langfuse, with the model, tokens, cost, input and output.

An owner or admin of the Langfuse organization sets it up once, under **AI Gateway** ([setup guide](https://langfuse.com/docs/gateway/get-started/admin-setup)):

1. In **Overview**, choose the project that the gateway records into, and set the ingestion mode to **Full**. The default mode records usage and cost without inputs and outputs, and Topics needs both to write a summary.
2. In **Connections & Models**, add a connection for the provider **OpenRouter** with an OpenRouter API key, and name it `openrouter`. The name is the prefix of the model IDs: `openrouter:<OpenRouter model ID>`.
3. In **Gateway keys**, create a key that is issued as a **Service account**, and set it as `LANGFUSE_GATEWAY_KEY`. With a service account key, the application can pass on the ID of its end user. A user key always records its owner as the user.

The Langfuse API keys from section 1 must belong to the project you chose in step 1.

### One trace per customer message

By default, the gateway records every model call as its own trace. The agent makes several calls to answer one customer message: one for each round of tool calls, and one for the reply. Topics analyzes traces, so the calls that answer one message should be one trace. Request headers take care of that, and of the attributes you filter by later:

| Header | Effect |
| --- | --- |
| `traceparent` | Calls with the same trace ID are recorded in one trace. `gateway_headers` creates a new ID for every customer message. |
| `langfuse-session-id` | Groups the traces of one conversation into a [session](https://langfuse.com/docs/observability/features/sessions) |
| `langfuse-user-id` | Sets the [user](https://langfuse.com/docs/observability/features/users) |
| `langfuse-environment` | Sets the [environment](https://langfuse.com/docs/observability/features/environments) |
| `langfuse-trace-name` | Names the trace |

`traceparent` is the [W3C Trace Context](https://www.w3.org/TR/trace-context/) header, with the format `00-<trace ID>-<parent ID>-01`. See [request headers](https://langfuse.com/docs/gateway/features/tracing#request-headers) for all headers the gateway reads.

<!-- CALLOUT_START type: "info" emoji: "ℹ️" -->
The gateway sees model calls only. To add your own steps to a trace, such as tool executions or retrieval, instrument the application with the Langfuse SDK and send the `traceparent` of your current span. See [trace context](https://langfuse.com/docs/gateway/features/tracing#trace-context).
<!-- CALLOUT_END -->

The next cell defines the headers and sends one short conversation through the gateway.

In [ ]:
import uuid


def gateway_headers(*, customer_id: str, session_id: str) -> tuple[str, dict]:
    """Headers for all model calls that answer one customer message. Returns the trace ID and the headers."""
    trace_id = uuid.uuid4().hex
    return trace_id, {
        "traceparent": f"00-{trace_id}-{uuid.uuid4().hex[:16]}-01",
        "langfuse-session-id": session_id,
        "langfuse-user-id": customer_id,
        "langfuse-environment": "production",
        "langfuse-trace-name": "answer-customer-message",
    }


session_id = f"chat-{uuid.uuid4().hex[:12]}"
messages = new_conversation("cus_1001")
for customer_text in ["Hi, when does my order arrive?", "Great, thanks. That is all I needed."]:
    messages.append({"role": "user", "content": customer_text})
    trace_id, headers = gateway_headers(customer_id="cus_1001", session_id=session_id)
    agent_reply(messages, client=app_client, extra_headers=headers)
    print(f"Customer: {customer_text}\nAssistant: {messages[-1]['content']}\nTrace: {langfuse.get_trace_url(trace_id=trace_id)}\n")

print("Tools used:", [call["tool"] for call in tool_log(messages)])

Open the trace links. Each trace holds one generation per model call. The input of a generation is everything that was sent to the model, so the last generation of a trace shows the whole exchange: the customer's message, the tool calls with their results, and the reply as its output.

### Generate production traffic

Real support conversations have several messages, so a second model plays the customer. Each simulated customer gets a persona card that is built from the demo data: a reason to write, a mood, and a way the chat ends. The agent never sees the card, and it is not stored in the traces, so whatever Topics reports later is discovered from the conversations themselves.

Every customer message becomes one trace, and all traces of a conversation share one session ID. The customer model calls OpenRouter directly, so its calls are not recorded in Langfuse.

<details>
<summary>Simulated customers: persona cards, the customer model and the conversation loop</summary>

In [ ]:
CUSTOMER_MODEL = "deepseek/deepseek-v4.1-flash-1017"
MAX_CUSTOMER_MESSAGES = 4
customer_client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=os.environ["OPENROUTER_API_KEY"])

REASON_SHARES = {"billing": 0.14, "refund": 0.10, "account": 0.16, "integration problem": 0.13, "integration setup": 0.04,
                 "integration request": 0.04, "cancel": 0.12, "order": 0.19, "off-topic": 0.08}
SYMPTOMS = {
    "LEDGERLY_TOKEN_EXPIRED": "Your sales stopped showing up in Ledgerly about a week ago. You changed nothing.",
    "LEDGERLY_TAX_MAPPING_MISSING": "You connected Ledgerly a few days ago, but the first sync never starts.",
    "SHOPFRONT_PLUGIN_OUTDATED": "Stock levels no longer sync between your Shopfront online store and Tallybird.",
    "SHOPFRONT_SKU_CONFLICT": "The product import from your Shopfront store stops halfway, every time.",
    "WEBHOOK_SIGNATURE_MISMATCH": "Your server receives the Tallybird webhooks, but your signature check fails on every one of them.",
    "WEBHOOK_PLAN_REQUIRED": "You want webhooks for new payments, but you cannot find the Webhooks page in the settings.",
}
ORDER_REASONS = {
    "processing": "You moved your shop and want order {order_id} delivered to the new address.",
    "shipped": "You want to know when order {order_id} ({item}) arrives. You need it for the weekend.",
    "delayed": "Order {order_id} ({item}) should have arrived on {expected_delivery} and is still not there.",
    "delivered": "Order {order_id} is marked as delivered, but nothing arrived at your shop.",
}
MOODS = [
    (0.38, "Neutral and to the point. You just want this done."),
    (0.12, "Friendly and grateful. You thank the assistant warmly for help."),
    (0.12, "Annoyed at first, because this has already cost you time. If the problem gets solved, you are relieved and say so."),
    (0.14, "Frustrated. You already tried what the help center says before you wrote. When the assistant suggests the same steps or asks for something you already said, you point that out sharply."),
    (0.10, "Not technical. You find the replies hard to follow and ask what a step or a term means."),
    (0.14, "In a hurry, with customers waiting. You complain about how long this takes and want the short version."),
]
ENDINGS = [
    (0.40, "When a reply solves your problem, confirm in one sentence that it is solved. Then the chat is over."),
    (0.24, "As soon as you have an answer, you leave without another message."),
    (0.12, "If the first reply does not solve your problem right away, ask for a human. Once a handoff is confirmed, the chat is over."),
    (0.12, "If the assistant asks you a question or asks you to do something, you leave without another message."),
    (0.12, "You keep asking until the problem is solved or the assistant says it cannot help."),
]


def reasons_for(customer: dict) -> dict[str, list[str]]:
    """What this customer could write about, per kind of request. The kind is never sent to the agent."""
    invoices, order = customer["invoices"], customer["order"]
    upgrade = {"Starter": "Plus", "Plus": "Pro", "Pro": "Plus"}[customer["plan"]]
    integrations = customer["integrations"]
    return {
        "billing": [
            f"You need a PDF copy of invoice {invoices[1]['invoice_id']} for your accountant.",
            "You want your VAT ID to appear on your invoices.",
            "Your card expires this month and you want to switch the payment method.",
            f"You think about moving to the {upgrade} plan and want to know what it costs per month.",
            f"Your latest invoice {invoices[2]['invoice_id']} is {invoices[2]['amount_usd']} USD. It used to be {invoices[0]['amount_usd']} USD. You want to know why.",
        ],
        "refund": [f"You were charged twice this month, invoices {invoices[2]['invoice_id']} and {invoices[3]['invoice_id']}. "
                   f"You want {invoices[3]['invoice_id']} refunded."] if len(invoices) > 3 else [],
        "account": [
            "The password reset email never arrives. You already checked the spam folder.",
            "You are locked out after typing a wrong password too often, and you need to open the shop.",
            "You want to rotate your API key and need to know if the old key stops working at once.",
            "You pushed your API key to a public code repository by accident.",
            "A former employee still knows the password. You want to change it and sign out all devices.",
        ],
        "integration problem": [SYMPTOMS[state["error"]] for state in integrations.values() if "error" in state],
        "integration setup": [f"You want to connect {name.capitalize()} for the first time and need the steps."
                              for name in ("ledgerly", "shopfront") if integrations[name]["status"] == "not_connected"],
        "integration request": [f"You want to connect Tallybird to {system} and ask how to set that up."
                                for system in ("the Bookwell accounting software", "your Cartello online store", "the Stampy loyalty app")],
        "cancel": [
            "You close your shop at the end of the month and want to cancel your plan.",
            "The app got too expensive for you and you want to cancel.",
            "You want to cancel, but you need your sales data for your tax return.",
            "You only open in summer and do not want to pay over the winter.",
        ],
        "order": [ORDER_REASONS[order["status"]].format(**order)],
        "off-topic": [
            "You want a cold brew recipe for your menu.",
            "You want help writing a job ad for a new shop assistant.",
            "You want to know which business bank account is best for a small shop.",
            "You want to know if you can deduct the card reader from your taxes.",
            "You want a short poem for the chalkboard in front of your shop.",
        ],
    }


def build_personas(count: int, seed: int = 11) -> list[dict]:
    rng = random.Random(seed)
    reasons_by_customer = [(customer, reasons_for(customer)) for customer in CUSTOMERS.values()]
    personas = []
    for _ in range(count):
        kind = rng.choices(list(REASON_SHARES), list(REASON_SHARES.values()))[0]
        customer, reasons = rng.choice([(customer, reasons[kind]) for customer, reasons in reasons_by_customer if reasons[kind]])
        personas.append({
            "customer_id": customer["customer_id"], "name": customer["name"], "business": customer["business"],
            "kind": kind,
            "reason": rng.choice(reasons),
            "mood": rng.choices([text for _, text in MOODS], [weight for weight, _ in MOODS])[0],
            "ending": rng.choices([text for _, text in ENDINGS], [weight for weight, _ in ENDINGS])[0],
            "language": rng.choices(["English", "German", "Spanish", "French"], [91, 3, 3, 3])[0],
        })
    return personas


CUSTOMER_PROMPT = """You play a customer in the support chat of Tallybird, a company that sells card readers and a point-of-sale app to small shops.

Who you are: {name}, owner of {business}.
Why you write: {reason}
Your mood: {mood}
How the chat ends for you: {ending}
Your language: you write in {language} and expect replies in {language}.

Write only the customer's next chat message: one to three sentences, in the style of a quick chat message. Your first message states why you write, with the invoice or order numbers from above if there are any. Never invent numbers, names or facts that are not given above.
When the chat is over for you, answer with exactly [END] and nothing else."""


def customer_message(persona: dict, messages: list[dict]) -> tuple[str | None, float]:
    """The customer's next message, or None when the customer leaves. Also returns the cost in USD."""
    chat = [f"{'You' if message['role'] == 'user' else 'Assistant'}: {message['content']}"
            for message in messages if message["role"] in ("user", "assistant") and message["content"]]
    response = customer_client.chat.completions.create(
        model=CUSTOMER_MODEL,
        messages=[
            {"role": "system", "content": CUSTOMER_PROMPT.format(**persona)},
            {"role": "user", "content": "\n\n".join(chat) + "\n\nYour next message:" if chat else "The chat is open. Write your first message."},
        ],
    )
    text = (response.choices[0].message.content or "").strip()
    cost = getattr(response.usage, "cost", None) or 0.0
    return (None if not text or "[END]" in text else text), cost


personas = build_personas(250)
print(len(personas), "personas, for example:")
for persona in personas[:3]:
    print(f"  {persona['name']}: {persona['reason']}")

In [ ]:
def run_conversation(persona: dict, *, client: OpenAI = app_client, model: str = DEFAULT_MODEL) -> dict:
    session_id = f"chat-{uuid.uuid4().hex[:12]}"
    messages = new_conversation(persona["customer_id"])
    result = {"session_id": session_id, "trace_ids": [], "messages": messages, "customer_messages": 0,
              "agent_cost": 0.0, "customer_cost": 0.0, "error": None}
    try:
        for _ in range(MAX_CUSTOMER_MESSAGES):
            text, cost = customer_message(persona, messages)
            result["customer_cost"] += cost
            if text is None:
                break
            messages.append({"role": "user", "content": text})
            result["customer_messages"] += 1
            trace_id, headers = gateway_headers(customer_id=persona["customer_id"], session_id=session_id)
            result["trace_ids"].append(trace_id)
            result["agent_cost"] += agent_reply(messages, client=client, model=model, extra_headers=headers)
    except Exception as error:
        result["error"] = repr(error)
    return result

</details>

The next cell plays 250 conversations, ten at a time.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=10) as pool:
    conversations = list(pool.map(run_conversation, personas))

print(f"{len(conversations)} conversations with {sum(conversation['customer_messages'] for conversation in conversations)} customer messages, one trace each")
print(f"OpenRouter cost: ${sum(conversation['agent_cost'] for conversation in conversations):.2f} for the agent, "
      f"${sum(conversation['customer_cost'] for conversation in conversations):.2f} for the simulated customers")

For this cookbook, the run produced 632 traces from 250 conversations in about seven minutes and cost $0.35 on OpenRouter. The traces appear in Langfuse within about a minute.

## 4. Add OpenRouter as an LLM connection

Topics calls models through the [LLM connections](https://langfuse.com/docs/administration/llm-connection) of your project, with your own key. OpenRouter works as a connection because it is OpenAI-compatible:

1. In Langfuse, open **Settings > LLM Connections** and add a new connection.
2. Select the **OpenAI** adapter.
3. Enter an OpenRouter API key.
4. Under **Advanced settings**, set the base URL to `https://openrouter.ai/api/v1`.
5. Add the OpenRouter model IDs you want to use as custom models: a fast text model for summaries, an embedding model, and a text model that names the topics.

This connection is separate from the gateway's OpenRouter connection in section 3. Topics calls OpenRouter through it directly, so its own model calls do not come back as new traces.

## 5. Set up Topics

Open **Topics** in your Langfuse project and start the setup. It has three steps:

1. **Set up topics:** Keep the four facets Intent, Sentiment, Outcome and Issues. Then select a model from the `openrouter` connection for each role: **Summaries**, **Embeddings** and **Grouping**. This cookbook uses `openai/gpt-6-luna`, `openai/text-embedding-3-small` and `openai/gpt-6.1-terra`. You cannot change the embedding model later.
2. **Select traces to analyze:** Add a filter on the environment `production`, so that the experiment runs of section 8 stay out of the analysis.
3. **Review:** Check the settings and select **Start analyzing**.

Topics summarizes up to 500 recent traces that match the filter, groups the summaries of each facet into topics, and names the topics. After that, it keeps analyzing new traces as they arrive. For the traffic of this cookbook, the first analysis cost about $0.35 on OpenRouter.

## 6. Explore the topics

The first topics appear a few minutes after the setup. On the Topics page, select a facet next to **Analyze**, and switch between **Treemap** and **Scatter**. Select a topic to read its description and to open its traces.

![Topics page with the topics of the Intent facet as a treemap](/images/cookbook/example-topics-openrouter/topics-intent-treemap.png)

Each facet groups the same traces from a different angle:

- **Intent:** what the customer wanted, for example "Delivery Status and Resolution" or "Cancel or Pause Subscription".
- **Sentiment:** how the customer felt about the replies.
- **Outcome:** how the exchange ended, for example "Refund Processed or Input Requested" or "Requested Content Not Provided".
- **Issues:** what went wrong in the handling, for example "Overpromised follow-up actions and outcomes".

Topics discovers these groups from the content of your traces. Your topic names and counts will differ from the ones in this notebook, even if you run the same code.

### Read the topics through the API

You can read everything on the Topics page through the API as well: the topics of each facet, and the current topic of each trace. Both endpoints take the same kind of `filter`, a JSON array of conditions.

<!-- CALLOUT_START type: "warning" emoji: "⚠️" -->
The Topics API lives under `/api/public/unstable/`. It is read-only, and its shape can change without notice.
<!-- CALLOUT_END -->

In [ ]:
import pandas as pd
import requests


def topics_api(path: str, **params) -> dict:
    """GET an endpoint of the unstable Topics API."""
    if "filter" in params:
        params["filter"] = json.dumps(
            [{"type": "stringOptions", "column": column, "operator": "any of", "value": values}
             for column, values in params["filter"].items()]
        )
    response = requests.get(
        f"{os.environ['LANGFUSE_BASE_URL']}/api/public/unstable/{path}",
        params=params,
        auth=(os.environ["LANGFUSE_PUBLIC_KEY"], os.environ["LANGFUSE_SECRET_KEY"]),
        timeout=60,
    )
    response.raise_for_status()
    return response.json()


def topic_assignments(**filter) -> list[dict]:
    """All traces with their current topic. A short page can still have a cursor, so follow it until it is empty."""
    rows, cursor = [], None
    while True:
        page = topics_api("topic-assignments", limit=100, filter=filter, **({"cursor": cursor} if cursor else {}))
        rows += page["data"]
        cursor = page["meta"]["cursor"]
        if not cursor:
            return rows


topics = pd.DataFrame(
    [{"facet": topic["facet"]["name"], "topic": topic["name"], "traces": topic["traceCount"]}
     for topic in topics_api("topics", limit=100)["data"]]
)
print(topics.groupby("facet")["topic"].count().rename("topics").to_string(), "\n")
print(topics[topics["facet"] == "Intent"].sort_values("traces", ascending=False)[["topic", "traces"]].to_string(index=False))

For the traffic of this cookbook, Topics found 19 topics for Intent, 15 for Outcome, 9 for Sentiment and 6 for Issues. The largest intent topics are "Delivery Status and Resolution" with 56 traces and "Cancel or Pause Subscription" with 48.

### Find an opportunity

Intent and Outcome are separate facets, so you can combine them: for each kind of request, how does the exchange usually end? The next cell reads the topics of every trace, with one column per facet, and prints the most common outcome topic for each intent topic. A trace that fits no topic of a facet is an outlier there, and the table leaves it out.

In [ ]:
assignments = pd.DataFrame(
    [{"trace_id": row["traceId"], "facet": row["facetName"], "topic": row["topic"]["name"]}
     for row in topic_assignments(status=["assigned"])]
)
per_trace = assignments.pivot_table(index="trace_id", columns="facet", values="topic", aggfunc="first")

outcomes = pd.crosstab(per_trace["Intent"], per_trace["Outcome"])
print(pd.DataFrame({
    "traces": outcomes.sum(axis=1),
    "most common outcome": outcomes.idxmax(axis=1),
    "share": (outcomes.max(axis=1) / outcomes.sum(axis=1)).round(2),
}).sort_values("traces", ascending=False).to_string())

A part of the result for the traffic of this cookbook:

| Intent topic | Traces | Most common outcome topic | Share |
| --- | --- | --- | --- |
| Delivery Status and Resolution | 54 | Delivery and Tracking Updates | 41% |
| Cancel or Pause Subscription | 45 | Cancellation or pause guidance | 58% |
| Duplicate Charge Refund | 22 | Refund Processed or Input Requested | 64% |
| Webhook Setup and Troubleshooting | 19 | Information and Guidance Provided | 84% |
| Change Order Delivery Address | 14 | Support Steps Without Direct Resolution | 86% |

Most intent topics end in an outcome topic that fits the request. Requests in "Duplicate Charge Refund" mostly end in "Refund Processed or Input Requested", and requests in "Webhook Setup and Troubleshooting" in "Information and Guidance Provided".

"Change Order Delivery Address" is different: 12 of its 14 traces end in "Support Steps Without Direct Resolution". Open the topic on the Topics page and read a few of its traces before you draw a conclusion. A typical one goes like this: the assistant looks up the order, sees that it is still processing, tells the customer that the address can still be changed, and opens a ticket for the human support team.

![The topic Change Order Delivery Address with its description and traces](/images/cookbook/example-topics-openrouter/topics-opportunity-topic.png)

The assistant does what it was built to do, so no error and no evaluator points at these traces. Still, each of these requests becomes work for a human, because the assistant has no tool to change an address. This is an opportunity: customers already try to do this through the assistant, and one more tool would let it finish the request in the chat. With the helpers from section 2, the tool could look like this:

```python
def update_delivery_address(customer_id: str, order_id: str, address: str) -> str:
    """Change the delivery address of an order that has not shipped yet."""
    ...


tool_schema(update_delivery_address, "Change the delivery address of a hardware order. Only possible while the order is processing.",
            customer_id=CUSTOMER_ID, order_id="Order ID, for example ORD-70310", address="The new delivery address")
```

This cookbook does not build the tool. The next sections go through all steps for a second finding, a failure.

## 7. Build a dataset from a topic

The Issues facet groups the traces by what went wrong in them. Each facet is clustered on its own, so a problem that is spread over many kinds of requests still shows up as one topic.

In [ ]:
issue_topics = topics_api("topics", limit=100, filter={"facetName": ["Issues"]})["data"]
for topic in sorted(issue_topics, key=lambda topic: -topic["traceCount"]):
    print(f"{topic['traceCount']:>4}  {topic['name']}: {topic['description']}")

For the traffic of this cookbook, the Issues facet has six topics:

| Issue topic | Traces |
| --- | --- |
| Mishandled requests and support steps | 83 |
| Unsupported Non-Troubleshooting Guidance | 66 |
| Overpromised follow-up actions and outcomes | 51 |
| Incorrect or unclear factual details | 26 |
| Speculative Claims and Inferences | 21 |
| Misleading or incomplete troubleshooting guidance | 20 |

Take "Unsupported Non-Troubleshooting Guidance". Topics describes it as "Answers included unverified instructions, options, conditions, or assurances in guidance outside troubleshooting." In its traces, the assistant gives instructions that no help center article contains. One reply tells the customer to go to "Settings → Security (or Profile → Sign Out)" and to look for an option like "Log out of all devices". The customer writes back: "Is there actually a button called 'Sign Out / Log Out All Devices' in Settings → Security — yes or no?"

![The Issues facet with the topic Unsupported Non-Troubleshooting Guidance selected](/images/cookbook/example-topics-openrouter/topics-issue-topic.png)

The trace of that reply shows the invented steps, and the topics of the trace next to the session and the user:

![A trace of the gateway with a reply that contains invented instructions](/images/cookbook/example-topics-openrouter/gateway-trace.png)

The system prompt tells the assistant to take product facts from the help center only. The model does not always follow that, and nothing fails when it does not. An evaluator would catch it, but nobody knew that this evaluator was needed.

The next cell collects the traces of the topic, and shows which intent topics they come from. Replace the topic name with one from your own project.

In [ ]:
ISSUE_TOPIC = "Unsupported Non-Troubleshooting Guidance"

dataset_trace_ids = sorted(row["traceId"] for row in topic_assignments(facetName=["Issues"], topicName=[ISSUE_TOPIC]))
intents = per_trace.loc[dataset_trace_ids, "Intent"]
print(f"{len(dataset_trace_ids)} traces in the topic, from {intents.nunique()} intent topics. The most common ones:")
print(intents.value_counts().head(5).to_string())

In this run, the topic has 66 traces, and they come from 15 different intent topics. The most common ones are "Sales Data Export Guidance" with 10 traces and "Integration Setup and Import Troubleshooting" with 9. A check of one kind of request would have seen only a small part of the problem.

Each trace is one customer message with the assistant's answer. The dataset item for a trace holds what the assistant received at that point: the conversation up to and including the customer's message. The gateway recorded it as the input of the first generation of the trace. `source_trace_id` links the item back to the production trace.

In [ ]:
DATASET_NAME = "support-assistant/invented-instructions"


def conversation(trace_id: str) -> list[dict]:
    """The messages that the assistant received for the customer message of this trace."""
    generations = langfuse.api.observations.get_many(
        trace_id=trace_id, type="GENERATION", fields="core,basic,io", limit=50
    ).data
    first = min(generations, key=lambda generation: generation.start_time)
    request = json.loads(first.input) if isinstance(first.input, str) else first.input
    return request["messages"]


langfuse.create_dataset(
    name=DATASET_NAME,
    description="Customer messages from one Topics issue topic, each with the conversation before it.",
    metadata={"issue_topic": ISSUE_TOPIC},
)
for trace_id in dataset_trace_ids:
    langfuse.create_dataset_item(
        dataset_name=DATASET_NAME,
        id=f"topics-openrouter-{trace_id}",
        input={"messages": conversation(trace_id)},
        source_trace_id=trace_id,
        metadata={"issue_topic": ISSUE_TOPIC},
    )
print(f"{len(dataset_trace_ids)} items in dataset '{DATASET_NAME}'")

## 8. Compare models on the dataset

The dataset holds real conversations in which the current model invented instructions. Run the same agent on it with several OpenRouter models, one [experiment](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk) per model. Each run writes the next reply of the conversation, with the same tools as in production.

The calls go through the gateway, as in production. Only the model ID changes. Two headers differ:

- `traceparent` carries the IDs of the current experiment step. The Langfuse SDK creates a trace for every dataset item, and with this header the gateway records its generations inside that trace. This is the combination of the SDK and the gateway from the note in section 3.
- `langfuse-environment` is `experiment`, so that these calls stay out of the Topics analysis of production.

In [ ]:
import asyncio
import time

from langfuse import Evaluation

CANDIDATE_MODELS = [
    DEFAULT_MODEL,
    "openrouter:meta/muse-ember",
    "openrouter:anthropic/claude-haiku-5.5",
    "openrouter:deepseek/deepseek-v4.1-flash-1017",
]
JUDGE_MODEL = "openrouter:openai/gpt-6-luna"


def experiment_headers() -> dict:
    """Headers for a gateway call that belongs to the current experiment step."""
    return {
        "traceparent": f"00-{langfuse.get_current_trace_id()}-{langfuse.get_current_observation_id()}-01",
        "langfuse-environment": "experiment",
    }


def agent_task(model: str):
    def next_reply(item) -> dict:
        started = time.time()
        messages = list(item.input["messages"])
        history = len(messages)
        cost = agent_reply(messages, client=app_client, model=model, extra_headers=experiment_headers())
        return {"reply": messages[-1]["content"], "tool_messages": messages[history:-1],
                "cost_usd": cost, "latency_s": round(time.time() - started, 1)}

    async def task(*, item, **kwargs):
        return await asyncio.to_thread(next_reply, item)  # a thread per item, so that items run in parallel

    return task

The production traces have no expected output, so the evaluator is an LLM judge without a reference. Its question comes from the topic: does the reply tell the customer to use something that no tool result mentions? The judge reads the conversation with all tool calls and results, and the new reply.

In [ ]:
JUDGE_PROMPT = """You review one reply of the support assistant of Tallybird, a point-of-sale product. The assistant knows how the product works only from tool results: help center articles, the customer's account and integration status.

Question: does the final reply tell the customer to use something that no tool result in the conversation mentions? "Something" is a menu path, screen, tab, button, field, option or command, in Tallybird or in another product.

Score 0 if it does. This also holds when the reply words it as a guess, for example "usually" or "you should see".
Score 1 if every such instruction appears in a tool result, or if the reply gives none.

Ignore everything else: offers to open a ticket, statements about what the support team can do, general advice, account facts, and explanations of words.

In the reasoning, quote each invented instruction, or write "none".

Conversation, with tool calls and results:
{conversation}

Final reply to review:
{reply}"""

JUDGE_SCHEMA = {"type": "json_schema", "json_schema": {"name": "verdict", "strict": True, "schema": {
    "type": "object", "additionalProperties": False, "required": ["reasoning", "score"],
    "properties": {"reasoning": {"type": "string"}, "score": {"type": "integer", "enum": [0, 1]}},
}}}


def transcript(messages: list[dict]) -> str:
    """The conversation as text, without the system prompt."""
    lines = []
    for message in messages:
        if message["role"] == "user":
            lines.append(f"Customer: {message['content']}")
        elif message["role"] == "tool":
            lines.append(f"Tool result: {message['content']}")
        elif message["role"] == "assistant":
            lines += [f"Assistant calls {call['function']['name']}({call['function']['arguments']})" for call in message.get("tool_calls") or []]
            if message.get("content"):
                lines.append(f"Assistant: {message['content']}")
    return "\n".join(lines)


def judge(input: dict, output: dict) -> Evaluation:
    response = app_client.chat.completions.create(
        model=JUDGE_MODEL,
        response_format=JUDGE_SCHEMA,
        messages=[{"role": "user", "content": JUDGE_PROMPT.format(
            conversation=transcript(input["messages"] + output["tool_messages"]), reply=output["reply"])}],
        extra_headers=experiment_headers(),
    )
    verdict = json.loads(response.choices[0].message.content)
    return Evaluation(name="no_invented_instructions", value=verdict["score"], comment=verdict["reasoning"])


async def no_invented_instructions(*, input, output, **kwargs):
    return await asyncio.to_thread(judge, input, output)

In [ ]:
dataset = langfuse.get_dataset(DATASET_NAME)
experiments = {}
for model in CANDIDATE_MODELS:
    experiments[model] = dataset.run_experiment(
        name=f"support-assistant on {model.removeprefix('openrouter:')}",
        description="Same agent and tools, different model. Dataset built from a Topics issue topic.",
        task=agent_task(model),
        evaluators=[no_invented_instructions],
        max_concurrency=8,
        metadata={"model": model},
    )
    print(f"{model}: {experiments[model].experiment_url}")
langfuse.flush()

In [ ]:
rows = []
for model, experiment in experiments.items():
    outputs = [result.output for result in experiment.item_results if result.output]
    scores = [evaluation.value for result in experiment.item_results
              for evaluation in result.evaluations if evaluation.name == "no_invented_instructions"]
    rows.append({
        "model": model.removeprefix("openrouter:"),
        "items": len(outputs),
        "no_invented_instructions": round(sum(scores) / len(scores), 2),
        "cost per reply ($)": round(sum(output["cost_usd"] for output in outputs) / len(outputs), 5),
        "latency (s)": round(sum(output["latency_s"] for output in outputs) / len(outputs), 1),
    })
print(pd.DataFrame(rows).to_string(index=False))

### Read the result

The cell prints one row per model. The result of this run:

| Model | Replies without invented instructions | Cost per reply | Latency |
| --- | --- | --- | --- |
| `liquid/lfm-3-1.7b` (current model) | 39% | $0.00048 | 4.7 s |
| `meta/muse-ember` | 82% | $0.00088 | 5.1 s |
| `anthropic/claude-haiku-5.5` | 71% | $0.00073 | 4.1 s |
| `deepseek/deepseek-v4.1-flash-1017` | 65% | $0.00047 | 4.0 s |

`meta/muse-ember` wrote a reply without invented instructions for 82% of the conversations, and the current model for 39%. A reply from `meta/muse-ember` costs almost twice as much. `deepseek/deepseek-v4.1-flash-1017` costs the same as the current model and reaches 65%.

Keep three things in mind when you read the table:

- The current model does not score 0. The dataset holds the conversations in which it invented instructions once, and it does not repeat every mistake when it answers the same message again.
- The judge is strict. It also counts a button that the reply names slightly differently than the help center does. It is equally strict with every model, so compare the models with each other and do not read a score as an exact error rate.
- The dataset has 66 items. A difference of a few points between two models can be chance.

Open the links from the experiments cell to read every reply with its score and the reasoning of the judge. The Experiments page lists the runs with their average score and latency. Your numbers will differ from the ones above.

![The four experiment runs on the Experiments page with their average score](/images/cookbook/example-topics-openrouter/experiments-comparison.png)

## 9. Close the loop

You started with traces and no idea what to look for. You now have two findings and a measured answer for one of them:

- **The opportunity:** customers ask the assistant to change a delivery address, and one more tool would let it do that.
- **The failure:** the assistant invents instructions. You know which model does it least, and what a reply from that model costs.

To act on the failure, change `DEFAULT_MODEL` to the model that won. If it costs more than you want to pay for every request, keep the default and send only the affected kinds of requests to the stronger model.

Then keep watching:

- Topics keeps analyzing new traces. Open the same issue topic a few days after the change and check whether it shrinks.
- The judge of section 8 is the evaluator that nobody knew was needed. Set it up as an [LLM-as-a-judge evaluator](https://langfuse.com/docs/evaluation/evaluation-methods/llm-as-a-judge) on production traces to track the score over time.
- The dataset stays in your project, so you can run the next model you are curious about against the same conversations.

### Related

- [Langfuse Topics](https://langfuse.com/docs/observability/features/topics)
- [Langfuse AI Gateway](https://langfuse.com/docs/gateway/overview)
- [Experiments via SDK](https://langfuse.com/docs/evaluation/experiments/experiments-via-sdk)
- [OpenRouter documentation](https://openrouter.ai/docs)